https://benfred.github.io/implicit/tutorial_lastfm.html#Getting-the-Dataset

### Downloading the data

In [ ]:
from implicit.datasets.lastfm import get_lastfm

artists, users, artist_user_plays = get_lastfm()

In [ ]:
artist_user_plays

### Training the model

In [ ]:
from implicit.nearest_neighbours import bm25_weight

# weight the matrix, both to reduce impact of users that have played the same artist thousands of times
# and to reduce the weight given to popular items
artist_user_plays = bm25_weight(artist_user_plays, K1=100, B=0.8)

# get the transpose since the most of the functions in implicit expect (user, item) sparse matrices instead of (item, user)
user_plays = artist_user_plays.T.tocsr()

In [ ]:
from implicit.als import AlternatingLeastSquares

model = AlternatingLeastSquares(factors=64, regularization=0.05, alpha=2.0)
model.fit(user_plays) # training in 2 minutes

### Making Recommendations

In [ ]:
userid = 12345
ids, scores = model.recommend(userid, user_plays[userid], N=10, filter_already_liked_items=False)

In [ ]:
import numpy as np
import pandas as pd
pd.DataFrame({"artist": artists[ids], "score": scores, "already_liked": np.isin(ids, user_plays[userid].indices)})

In [ ]:
# making multiple
userids = np.arange(1000)
ids, scores = model.recommend(userids, user_plays[userids])
ids, ids.shape

### Converting the implicit model to torch

In [ ]:
# model.item_factors
# model.user_factors
# model.regularization

In [ ]:
item_factors = model.item_factors          # [num_sounds, 64]  (numpy)
user_factors = model.user_factors          # [num_users,  64]

# Precompute the fold-in matrix for new users
# YtY = item_factors.T @ item_factors
YtY = item_factors.T @ item_factors        # [64, 64]
reg = np.eye(64) * model.regularization
foldin_matrix = np.linalg.solve(YtY + reg, item_factors.T)  # [64, num_sounds]
# new_user_vec = foldin_matrix @ r  where r is [num_sounds] volume vector

In [ ]:
# **Wraps ad pytorch module**

import torch
import torch.nn as nn

class ALSRecommender(nn.Module):
    def __init__(self, item_factors: np.ndarray, foldin_matrix: np.ndarray):
        super().__init__()
        # Register as buffers (not trained, but exported with the model)
        self.register_buffer("item_factors",  torch.tensor(item_factors,  dtype=torch.float32))
        self.register_buffer("foldin_matrix", torch.tensor(foldin_matrix, dtype=torch.float32))

    def forward(self, interaction_vector: torch.Tensor) -> torch.Tensor:
        """
        Args:
            interaction_vector: [num_sounds] float tensor
                                 volume at index i if sound i is playing, else 0.0
        Returns:
            scores: [num_sounds] float tensor — higher = better recommendation
        """
        # Fold-in: derive user latent vector from current mix
        user_vec = self.foldin_matrix @ interaction_vector   # [64]

        # Score all sounds
        scores = self.item_factors @ user_vec                # [num_sounds]
        return scores


# Instantiate with pretrained weights
als_model = ALSRecommender(item_factors, foldin_matrix)
als_model.eval()

In [ ]:
# Build an interaction vector for a test mix
# e.g. rain(idx=34, vol=0.28), ocean(idx=17, vol=1.00)
num_sounds = item_factors.shape[0]
r = np.zeros(num_sounds, dtype=np.float32)
r[34] = 0.28
r[17] = 1.00

# implicit's own recommendation (for reference)
# implicit uses a different internal path for known users, 
# so compare against the fold-in math directly
expected_user_vec = foldin_matrix @ r
expected_scores   = item_factors @ expected_user_vec

# PyTorch model
r_tensor = torch.tensor(r)
with torch.no_grad():
    pt_scores = als_model(r_tensor)

np.testing.assert_allclose(pt_scores.numpy(), expected_scores, rtol=1e-5)
print("✓ Outputs match")

#### Export to LiteRT using TensorFlow

In [ ]:
# # **export to litert**
# import ai_edge_torch  # pip install ai-edge-torch

# sample_input = torch.zeros(num_sounds)  # [num_sounds]

# edge_model = ai_edge_torch.convert(als_model, (sample_input,))
# edge_model.export("als_recommender.tflite")

In [ ]:
import tensorflow as tf

# Build equivalent Keras model
inp = tf.keras.Input(shape=(num_sounds,), name="interaction_vector")
user_vec = tf.keras.layers.Dense(
    64, use_bias=False, name="foldin",
    kernel_initializer=tf.constant_initializer(foldin_matrix.T)
)(inp)
scores = tf.keras.layers.Dense(
    num_sounds, use_bias=False, name="item_scores",
    kernel_initializer=tf.constant_initializer(item_factors)
)(user_vec)

keras_model = tf.keras.Model(inputs=inp, outputs=scores)
keras_model.trainable = False  # weights are frozen ALS factors

# Convert
converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open("als_recommender.tflite", "wb") as f:
    f.write(tflite_model)

#### Export to LiteRT using Torch -> ONNX -> Tensorflow SavedModel -> TFLite

In [ ]:
sample_input = torch.zeros(num_sounds)

torch.onnx.export(
    als_model,
    sample_input,
    "model.onnx",
    export_params=True,
    opset_version=11,        # Use 11–13 for best TF compatibility
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}
)

In [ ]:
import onnx
from onnx_tf.backend import prepare

# Load the ONNX model
onnx_model = onnx.load("model.onnx")

# Convert to TensorFlow
tf_rep = prepare(onnx_model)

# Export as SavedModel
tf_rep.export_graph("saved_model")

In [ ]:
# Convert TensorFlow SavedModel -> TFLite

import tensorflow as tf

# Load the SavedModel
converter = tf.lite.TFLiteConverter.from_saved_model("saved_model")

# Optional: Enable optimizations (quantization)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Convert
tflite_model = converter.convert()

# Save the .tflite file
with open("model.tflite", "wb") as f:
    f.write(tflite_model)

print("Conversion complete!")

In [ ]:
## verify TFLite Model
import numpy as np
import tensorflow as tf

# Load and test the TFLite model
interpreter = tf.lite.Interpreter(model_path="model.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Run inference
test_input = np.random.randn(1, 3, 224, 224).astype(np.float32)
interpreter.set_tensor(input_details[0]['index'], test_input)
interpreter.invoke()

output = interpreter.get_tensor(output_details[0]['index'])
print("Output shape:", output.shape)